In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [2]:
class_categori = {
    '0': 'Appetizer',
    '1': 'Beverages',
    '2': 'Breakfast',
    '3': 'Desser',
    '4': 'Main Dish',
    '5': 'Other',
    '6': 'Side Dishes',
    '7': 'Soup'
}

categori_class = {
    'Appetizer': 0,
    'Beverages': 1,
    'Breakfast': 2,
    'Dessert': 3,
    'Main Dish': 4,
    'Other': 5,
    'Side Dishes': 6,
    'Soup': 7
}

In [3]:
# Modelin çıkış katmanını çok sınıflı olarak değiştir
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(100, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.output = nn.Linear(64, 8)  # 8 sınıf için

        self.dropout = nn.Dropout(0.5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.dropout(x)
        x = self.relu(self.fc4(x))
        return self.output(x)  # Softmax eklemiyoruz, CrossEntropyLoss bunu halledecek


In [4]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        """
        patience (int): Kayıp belirli bir epoch sayısı boyunca iyileşmezse durdurur.
        min_delta (float): Kayıptaki minimum iyileşme miktarı.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0  # Eğer iyileşme varsa sıfırla
        else:
            self.counter += 1  # İyileşme yoksa artır
            if self.counter >= self.patience:
                self.early_stop = True

In [5]:
import pandas as pd

df = pd.read_parquet('./preprocessed_data.parquet')

In [6]:
import numpy as np

X = np.stack(df['tfidf'].to_numpy())
y = np.array([categori_class[category] for category in df['categories'].to_numpy()])

In [7]:
# Örneğin, X ve y veri setleri NumPy array formatında olduğunu varsayalım.
# Standartlaştırma
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# PyTorch tensörlerine çevir
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)  # Binary classification için float

# Veri setini eğitim ve doğrulama olarak ayır
train_size = int(0.8 * len(X_train_tensor))
val_size = len(X_train_tensor) - train_size
train_dataset, val_dataset = random_split(TensorDataset(X_train_tensor, y_train_tensor), [train_size, val_size])

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1024, num_workers=4, pin_memory=True)


In [ ]:
model = SimpleNN()

# Ağırlık başlatma fonksiyonu
def initialize_weights(m):
    if isinstance(m, nn.Linear):
        # Örneğin, He Başlatma
        torch.nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)

# Ağırlık başlatma fonksiyonunu modele uygula
model.apply(initialize_weights)

criterion = nn.CrossEntropyLoss()  # Binary Cross-Entropy Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [9]:
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

# Cihaz (GPU kullanımı)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Early Stopping ayarı
early_stopping = EarlyStopping(patience=5, min_delta=0.01)

# Mixed Precision için GradScaler
scaler = GradScaler()

num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    # Eğitim döngüsü
    for inputs, labels in tqdm(train_loader, desc='Training epoch...', leave=False):
        inputs, labels = inputs.to(device), labels.to(device).long()
        
        optimizer.zero_grad()
        
        # Mixed Precision ile ileri yayılım
        with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)
        
        # Geri yayılım ve optimizasyon adımları
        scaler.scale(loss).backward()  # Loss'u scaler ile ölçekliyoruz
        scaler.step(optimizer)         # Scale edilmiş optimizasyon adımı
        scaler.update()                # Scaler'i güncelle
        
        running_loss += loss.item()
        
        # Tahminleri al ve doğru sınıfları kontrol et
        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)
    
    train_accuracy = 100 * correct_train / total_train

    # Validation (Doğrulama) döngüsü
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device).long()
            
            # Mixed Precision ile doğrulama
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Tahminleri al ve doğru sınıfları kontrol et
            _, predicted = torch.max(outputs, 1)
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)

    val_accuracy = 100 * correct_val / total_val

    # Sonuçları ekrana yazdır
    tqdm.write(f"Epoch [{epoch+1}/{num_epochs}], "
           f"Train Loss: {running_loss/len(train_loader):.4f}, "
           f"Train Accuracy: {train_accuracy:.2f}%, "
           f"Val Loss: {val_loss/len(val_loader):.4f}, "
           f"Val Accuracy: {val_accuracy:.2f}%")

    # Early stopping kontrolü
    early_stopping(val_loss/len(val_loader))
    if early_stopping.early_stop:
        print("Early stopping ile eğitim durduruldu.")
        break


Epoch [1/50], Train Loss: 0.8854, Train Accuracy: 65.95%, Val Loss: 0.6931, Val Accuracy: 71.57%


Epoch [2/50], Train Loss: 0.7385, Train Accuracy: 70.54%, Val Loss: 0.6478, Val Accuracy: 73.90%


Epoch [3/50], Train Loss: 0.7009, Train Accuracy: 72.07%, Val Loss: 0.6189, Val Accuracy: 75.10%


Epoch [4/50], Train Loss: 0.6755, Train Accuracy: 73.30%, Val Loss: 0.5962, Val Accuracy: 76.60%


Epoch [5/50], Train Loss: 0.6593, Train Accuracy: 74.04%, Val Loss: 0.5725, Val Accuracy: 77.82%


...

Epoch [20/50], Train Loss: 0.5669, Train Accuracy: 78.57%, Val Loss: 0.4834, Val Accuracy: 82.15%


Epoch [21/50], Train Loss: 0.5638, Train Accuracy: 78.68%, Val Loss: 0.4788, Val Accuracy: 82.24%


Epoch [22/50], Train Loss: 0.5629, Train Accuracy: 78.82%, Val Loss: 0.4775, Val Accuracy: 82.25%


Epoch [23/50], Train Loss: 0.5605, Train Accuracy: 78.88%, Val Loss: 0.4783, Val Accuracy: 82.18%


Epoch [24/50], Train Loss: 0.5580, Train Accuracy: 78.98%, Val Loss: 0.4739, Val Accuracy: 82.49%
Early stopping ile eğitim durduruldu.


In [ ]:
torch.save(model.state_dict, './model_checkpoints/model_v3.pth')

In [11]:
# PyTorch tensörlerine çevir
X_test_tensor = torch.tensor(X_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_train, dtype=torch.float32)  # Binary classification için float

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# DataLoader
test_loader = DataLoader(test_dataset, batch_size=64)


In [12]:
# Modeli test seti üzerinde değerlendirme
model.eval()  # Modeli evaluation moduna al
test_loss = 0.0
correct_test = 0
total_test = 0

with torch.no_grad():  # Test aşamasında grad hesaplamaya gerek yok
    for inputs, labels in tqdm(test_loader, desc='Calculating outputs...'):
        inputs, labels = inputs.to(device), labels.to(device).long()
        
        # Tahminleri al ve kaybı hesapla
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        
        # Tahmin edilen sınıfları al ve doğru sınıfları kontrol et
        _, predicted = torch.max(outputs, 1)
        correct_test += (predicted == labels).sum().item()
        total_test += labels.size(0)

# Test doğruluğunu ve kaybını hesapla
test_accuracy = 100 * correct_test / total_test
average_test_loss = test_loss / len(test_loader)

print(f"Test Loss: {average_test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")


Calculating outputs...: 100%|██████████| 17521/17521 [00:30<00:00, 581.59it/s]

Test Loss: 0.4610, Test Accuracy: 82.94%
